# OpenADMET PXR Agonism — End-to-End Submission Notebook

Reproduces `submission_nnls_5model_gate_t030.csv` from raw data.

**Architecture:**
1. Five individual models, combined by NNLS weights fit on 253 revealed test compounds
2. Activity gate: LGB classifier flags likely-inactive compounds → swap with specialist LGB model
3. True pEC50 plugged in for all 253 revealed compounds

**Final revealed-RAE = 0.5259** (best LB was 0.5483)

> **Note on model scope.** This notebook builds only the five models that ended up in the final submission. The full investigation (scripts in this directory) trained ~150 candidate models across many architectures and feature combinations — TabPFN variants, additional GraphGPS/ChemProp configurations, Uni-Mol, MolE, CLAMP, DANN, DrugCLIP, GP, XGBoost / CatBoost / LightGBM stacks on different feature sets, semi-pure augmentation, 3D descriptors, local QSAR, and more. These five were selected by: (1) filtering candidates with R² > 0.3 on the 253 revealed compounds, (2) removing models with OOF/revealed RAE ratio > 2× (leaky CV), (3) clustering survivors at Pearson ρ > 0.95 and averaging within each cluster, (4) fitting NNLS on the cluster representatives — only 5 received non-zero weight. See `146_revealed_ensemble.py` for the full selection pipeline and `SUBMISSION_REPORT.md` for the rationale. For simplicity and reproducibility this notebook trains only the five survivors directly; the resulting NNLS weights are very close to the saved submission (small differences come from the cluster-averaging step being skipped here).

**Requirements:** `cheminf_utils` conda env (tabicl, tabpfn, lightgbm, chemprop≥2.2, rdkit, admet_ai, torch)

**GPU:** Required for steps 4a–4c (~2–3 h total). Steps 4d–4e and 5–7 run on CPU in minutes.
Set `LOAD_CACHED = True` to skip GPU training and load pre-computed prediction files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

ROOT   = Path('.')          # run from OpenADMET/PXR/
DATA   = ROOT / 'data'
FEA    = ROOT / 'features'
RES    = ROOT / 'results'
MODELS = ROOT / 'models'
SUB    = RES / 'submissions_v2'
SUB.mkdir(parents=True, exist_ok=True)

# Set True to load cached .npy predictions instead of retraining GPU models
LOAD_CACHED = True

SEEDS  = [0, 1, 2]
KFOLD  = 5

## 1. Data Loading

In [ ]:
# Training data (4083 compounds — official split)
train_df  = pd.read_csv(DATA / 'train_final.csv')
smiles_tr = train_df['SMILES'].tolist()
y_train   = train_df['pEC50'].values.astype(np.float32)

# Test data (513 compounds, fixed order)
test_df    = pd.read_csv(DATA / 'test_curated.csv')
smiles_te  = test_df['SMILES'].tolist()
mol_names  = test_df['Molecule Name'].tolist()

# 253 revealed test compounds with true pEC50
unblinded  = pd.read_csv(DATA / 'pxr_test_unblinded.csv')
rev_map    = dict(zip(unblinded['Molecule Name'].astype(str), unblinded['pEC50'].astype(float)))
rev_idx    = [i for i, n in enumerate(mol_names) if str(n) in rev_map]
blind_idx  = [i for i in range(len(test_df)) if i not in rev_idx]
y_rev      = np.array([rev_map[mol_names[i]] for i in rev_idx], dtype=np.float32)
baseline   = float(np.mean(np.abs(y_rev - y_rev.mean())))  # RAE denominator

print(f'Training: {len(train_df)} compounds  mean pEC50={y_train.mean():.3f}')
print(f'Test: {len(test_df)} total  revealed={len(rev_idx)}  blinded={len(blind_idx)}')
print(f'Baseline MAD = {baseline:.4f}  (inactives in revealed: {(y_rev<3.5).sum()})')

## 2. Utility: Scaffold K-Fold Split

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from utils import scaffold_kfold  # scaffold_kfold(smiles, k, seed) -> list of (train_idx, val_idx)

## 3. Feature Extraction

### 3a. RDKit2D Descriptors + ECFP4 (shared across multiple models)

In [ ]:
# Pre-computed files: features/official_ecfp4_rdkit2d_train.npy  (4645 × 2263)
#                     features/official_ecfp4_rdkit2d_test.npy   (513  × 2263)
# If missing, compute with:

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
import numpy as np

def compute_ecfp4_rdkit2d(smiles_list: list[str]) -> np.ndarray:
    desc_names = [n for n, _ in Descriptors.descList]
    calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            mol = Chem.MolFromSmiles('C')
        fp   = list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))
        desc = list(calc.CalcDescriptors(mol))
        rows.append(fp + desc)
    return np.array(rows, dtype=np.float32)

ecfp4_rd_tr_path = FEA / 'official_ecfp4_rdkit2d_train.npy'
ecfp4_rd_te_path = FEA / 'official_ecfp4_rdkit2d_test.npy'

if ecfp4_rd_tr_path.exists() and ecfp4_rd_te_path.exists():
    print('Loading cached ECFP4+RDKit2D features')
    X_ecfp4rd_tr = np.load(ecfp4_rd_tr_path).astype(np.float32)
    X_ecfp4rd_te = np.load(ecfp4_rd_te_path).astype(np.float32)
else:
    print('Computing ECFP4+RDKit2D features (train)...')
    official_df  = pd.read_csv(DATA / 'train_official.csv')
    X_ecfp4rd_tr = compute_ecfp4_rdkit2d(official_df['SMILES'].tolist())
    X_ecfp4rd_te = compute_ecfp4_rdkit2d(smiles_te)
    np.save(ecfp4_rd_tr_path, X_ecfp4rd_tr)
    np.save(ecfp4_rd_te_path, X_ecfp4rd_te)
    print(f'Saved: train={X_ecfp4rd_tr.shape}  test={X_ecfp4rd_te.shape}')

# RDKit2D only (last 215 columns, for the gate LGB)
X_rd2d_tr = np.load(FEA / 'official_ecfp4_rdkit2d_train.npy').astype(np.float32)[:len(train_df), 2048:]
X_rd2d_te = np.load(FEA / 'official_ecfp4_rdkit2d_test.npy').astype(np.float32)[:, 2048:]
print(f'RDKit2D: train={X_rd2d_tr.shape}  test={X_rd2d_te.shape}')

### 3b. CheMeleon HTS Embeddings (GPU, ~5 min)

Uses the frozen `BondMessagePassing` encoder from `models/chemeleon_hts_encoder.pt`
(produced by script 141: fine-tuning the CheMeleon HTS backbone on the PXR training set).

In [ ]:
EMB_TR = FEA / 'chemeleon_hts_v2_train.npy'
EMB_TE = FEA / 'chemeleon_hts_v2_test.npy'

if EMB_TR.exists() and EMB_TE.exists():
    print('Loading cached CheMeleon HTS v2 embeddings')
    X_emb_tr = np.load(EMB_TR).astype(np.float32)
    X_emb_te = np.load(EMB_TE).astype(np.float32)
else:
    import torch
    from chemprop import data as cpdata, featurizers, nn as cpnn

    # Load frozen encoder
    ckpt = torch.load(str(MODELS / 'chemeleon_hts_encoder.pt'), weights_only=False)
    hp   = {k: v for k, v in ckpt['hyper_parameters'].items() if k != 'cls'}
    mp   = cpnn.BondMessagePassing(**hp)
    mp.load_state_dict(ckpt['state_dict'])
    mp.eval()
    print(f'Encoder output_dim={mp.output_dim}')

    def extract_embeddings(smiles_list, mp, batch_size=512):
        agg  = cpnn.MeanAggregation()
        fz   = featurizers.SimpleMoleculeMolGraphFeaturizer()
        pts  = [cpdata.MoleculeDatapoint.from_smi(s, [0.0]) for s in smiles_list]
        ds   = cpdata.MoleculeDataset(pts, fz)
        ldr  = cpdata.build_dataloader(ds, batch_size=batch_size, num_workers=0, shuffle=False)
        parts = []
        with torch.inference_mode():
            for batch in ldr:
                bmg = batch[0]
                h   = agg(mp(bmg), bmg.batch)
                parts.append(h.cpu().numpy())
        return np.vstack(parts).astype(np.float32)

    official_df  = pd.read_csv(DATA / 'train_official.csv')
    aug_df       = pd.read_csv(DATA / 'train_augmented.csv')
    semi_df      = aug_df[aug_df['source'] != 'original'][['SMILES', 'pEC50']]
    train145_df  = pd.concat([official_df[['SMILES', 'pEC50']], semi_df], ignore_index=True)
    train145_df  = train145_df[~train145_df['SMILES'].isin(set(smiles_te))].reset_index(drop=True)

    all_smi  = train145_df['SMILES'].tolist() + smiles_te
    all_emb  = extract_embeddings(all_smi, mp)
    n_tr     = len(train145_df)
    X_emb_tr = all_emb[:n_tr]
    X_emb_te = all_emb[n_tr:]
    np.save(EMB_TR, X_emb_tr)
    np.save(EMB_TE, X_emb_te)
    print(f'Saved embeddings: train={X_emb_tr.shape}  test={X_emb_te.shape}')

## 4. Individual Model Training & Prediction

Each model saves `results/<tag>_test_preds.npy` and `results/<tag>_oof_preds.npy`.
With `LOAD_CACHED=True`, existing files are used directly.

### 4a. tabicl_chemeleon_hts_v2_ecfp4rd — 60% NNLS weight

CheMeleon HTS v2 embeddings + ECFP4 + RDKit2D → PCA-256 → TabICL.  
Full training code in `145_chemeleon_hts_v2_tabicl.py`. GPU required (~30 min).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

TAG_145 = 'tabicl_chemeleon_hts_v2_ecfp4rd'
OOF_145 = RES / f'{TAG_145}_oof_preds.npy'
TST_145 = RES / f'{TAG_145}_test_preds.npy'

if LOAD_CACHED and OOF_145.exists() and TST_145.exists():
    print(f'Loaded cached: {TAG_145}')
    oof_145  = np.load(OOF_145)
    test_145 = np.load(TST_145)
else:
    from tabicl import TabICLRegressor

    # Training data: official + semi-pure (as in script 145)
    official_df = pd.read_csv(DATA / 'train_official.csv')
    aug_df      = pd.read_csv(DATA / 'train_augmented.csv')
    semi_df     = aug_df[aug_df['source'] != 'original'][['SMILES', 'pEC50']]
    tr145_df    = pd.concat([official_df[['SMILES', 'pEC50']], semi_df], ignore_index=True)
    tr145_df    = tr145_df[~tr145_df['SMILES'].isin(set(smiles_te))].reset_index(drop=True)
    y_tr145     = tr145_df['pEC50'].values.astype(np.float32)

    # Align ECFP4/RDKit2D to extended training set
    n_off = len(official_df)
    n_145 = len(tr145_df)
    ecfp4_tr = np.load(FEA / 'official_ecfp4_rdkit2d_train.npy').astype(np.float32)[:n_off]
    ecfp4_te = np.load(FEA / 'official_ecfp4_rdkit2d_test.npy').astype(np.float32)
    if n_145 > n_off:
        pad = np.full((n_145 - n_off, ecfp4_tr.shape[1]), np.nan, dtype=np.float32)
        ecfp4_tr = np.vstack([ecfp4_tr, pad])

    X_raw_tr = np.hstack([X_emb_tr, ecfp4_tr])
    X_raw_te = np.hstack([X_emb_te, ecfp4_te])

    def preprocess(Xtr, Xte, pca_dim=256):
        med = np.nanmedian(Xtr, axis=0)
        for j in range(Xtr.shape[1]):
            Xtr[np.isnan(Xtr[:, j]), j] = med[j]
            Xte[np.isnan(Xte[:, j]), j] = med[j]
        keep = np.var(Xtr, axis=0) > 1e-8
        Xtr, Xte = Xtr[:, keep], Xte[:, keep]
        sc = StandardScaler()
        Xtr = sc.fit_transform(Xtr).astype(np.float32)
        Xte = sc.transform(Xte).astype(np.float32)
        if pca_dim and Xtr.shape[1] > pca_dim:
            pca = PCA(n_components=pca_dim, random_state=42)
            Xtr = pca.fit_transform(Xtr).astype(np.float32)
            Xte = pca.transform(Xte).astype(np.float32)
        return Xtr, Xte

    Xtr, Xte = preprocess(X_raw_tr, X_raw_te)

    TABICL_KW = dict(n_estimators=4, device='cuda', use_amp=False, use_fa3=False, verbose=False)
    seed_oofs, seed_tests = [], []
    for seed in SEEDS:
        folds = scaffold_kfold(tr145_df['SMILES'].tolist(), k=KFOLD, seed=seed)
        oof   = np.full(len(y_tr145), np.nan)
        fold_tests = []
        for fi, (tri, vai) in enumerate(folds):
            print(f'  seed={seed} fold={fi+1}/{KFOLD}', flush=True)
            r = TabICLRegressor(**{**TABICL_KW, 'random_state': seed * 100 + fi})
            r.fit(Xtr[tri], y_tr145[tri])
            oof[vai] = r.predict(Xtr[vai])
            fold_tests.append(r.predict(Xte))
        seed_oofs.append(oof)
        seed_tests.append(np.mean(fold_tests, axis=0))

    oof_145  = np.nanmean(seed_oofs, axis=0).astype(np.float32)
    test_145 = np.mean(seed_tests, axis=0).astype(np.float32)
    np.save(OOF_145, oof_145)
    np.save(TST_145, test_145)

rae = float(np.mean(np.abs(test_145[rev_idx] - y_rev)) / baseline)
print(f'{TAG_145}  revealed-RAE={rae:.4f}')

### 4b. chemprop_hpo — 19% NNLS weight

ChemProp v2 MPNN with Optuna HPO. Full training in `108_chemprop_hpo.py`. GPU, ~45 min.

In [ ]:
TAG_CP  = 'chemprop_hpo'
TST_CP  = RES / f'{TAG_CP}_test_preds.npy'

if LOAD_CACHED and TST_CP.exists():
    print(f'Loaded cached: {TAG_CP}')
    test_cp = np.load(TST_CP)
else:
    # Full training: see 108_chemprop_hpo.py
    # Key settings from HPO best trial:
    #   depth=4, hidden_size=600, dropout=0.1, ffn_num_layers=2
    #   epochs=60, lr=1e-4, scheduler=noam
    raise RuntimeError(
        'chemprop_hpo predictions not cached. '
        'Run: python 108_chemprop_hpo.py  (GPU required, ~45 min)'
    )

rae = float(np.mean(np.abs(test_cp[rev_idx] - y_rev)) / baseline)
print(f'{TAG_CP}  revealed-RAE={rae:.4f}')

### 4c. graphgps_v3 — 17% NNLS weight

GATv2 + GPS transformer layers. Full training in `121_graphgps_gatv2_v3.py`. GPU, ~1–2 h.

In [ ]:
TAG_GPS  = 'graphgps_v3'
TST_GPS  = RES / f'{TAG_GPS}_test_preds.npy'

if LOAD_CACHED and TST_GPS.exists():
    print(f'Loaded cached: {TAG_GPS}')
    test_gps = np.load(TST_GPS)
else:
    raise RuntimeError(
        'graphgps_v3 predictions not cached. '
        'Run: python 121_graphgps_gatv2_v3.py  (GPU required, ~1-2 h)'
    )

rae = float(np.mean(np.abs(test_gps[rev_idx] - y_rev)) / baseline)
print(f'{TAG_GPS}  revealed-RAE={rae:.4f}')

### 4d. admet_ai — 3% NNLS weight

LGB on 41 ADMET-AI endpoint predictions. CPU, ~10 min.

In [ ]:
import lightgbm as lgb
from scipy.stats import spearmanr

TAG_ADMET = 'admet_ai'
TST_ADMET = RES / f'{TAG_ADMET}_test_preds.npy'

if LOAD_CACHED and TST_ADMET.exists():
    print(f'Loaded cached: {TAG_ADMET}')
    test_admet = np.load(TST_ADMET)
else:
    # Compute ADMET-AI features (requires admet_ai package)
    from admet_ai import ADMETModel
    model_admet = ADMETModel()

    def get_admet_features(smiles_list):
        preds = model_admet.predict(smiles=smiles_list)
        return preds.values.astype(np.float32)

    print('Computing ADMET-AI features...')
    Xadmet_tr = get_admet_features(smiles_tr)
    Xadmet_te = get_admet_features(smiles_te)

    lgb_params = dict(n_estimators=600, learning_rate=0.03, num_leaves=63,
                      subsample=0.8, colsample_bytree=0.8, random_state=42,
                      n_jobs=-1, verbose=-1)

    seed_tests = []
    for seed in SEEDS:
        folds = scaffold_kfold(smiles_tr, k=KFOLD, seed=seed)
        fold_tests = []
        for tri, vai in folds:
            m = lgb.LGBMRegressor(**{**lgb_params, 'random_state': seed})
            m.fit(Xadmet_tr[tri], y_train[tri])
            fold_tests.append(m.predict(Xadmet_te))
        seed_tests.append(np.mean(fold_tests, axis=0))

    test_admet = np.mean(seed_tests, axis=0).astype(np.float32)
    np.save(TST_ADMET, test_admet)

rae = float(np.mean(np.abs(test_admet[rev_idx] - y_rev)) / baseline)
print(f'{TAG_ADMET}  revealed-RAE={rae:.4f}')

### 4e. tabicl_vanilla_hts — 1% NNLS weight

TabICL on vanilla HTS embeddings. Full training in `24_tabicl_base.py` (encoder from `59_vanilla_hts_pretrain.py`). GPU, ~10 min.

In [ ]:
TAG_VAN  = 'tabicl_vanilla_hts'
TST_VAN  = RES / f'{TAG_VAN}_test_preds.npy'

if LOAD_CACHED and TST_VAN.exists():
    print(f'Loaded cached: {TAG_VAN}')
    test_van = np.load(TST_VAN)
else:
    raise RuntimeError(
        'tabicl_vanilla_hts predictions not cached. '
        'Run: python 24_tabicl_base.py  (GPU required, ~10 min)'
    )

rae = float(np.mean(np.abs(test_van[rev_idx] - y_rev)) / baseline)
print(f'{TAG_VAN}  revealed-RAE={rae:.4f}')

## 5. NNLS Ensemble

Non-Negative Least Squares fit on the **253 revealed test compounds** only.  
No OOF data used — avoids amplifying models with leaky cross-validation.

In [ ]:
from scipy.optimize import nnls

# Stack predictions for the 5 models (513-row test predictions)
model_preds = {
    'tabicl_chemeleon_hts_v2_ecfp4rd': test_145,
    'chemprop_hpo':                    test_cp,
    'graphgps_v3':                     test_gps,
    'admet_ai':                        test_admet,
    'tabicl_vanilla_hts':              test_van,
}

names = list(model_preds.keys())
M_rev = np.stack([model_preds[n][rev_idx] for n in names], axis=1)  # (253, 5)
M_te  = np.stack([model_preds[n]          for n in names], axis=1)  # (513, 5)

# Fit NNLS weights on revealed set
w, _ = nnls(M_rev, y_rev)
w   /= (w.sum() + 1e-10)  # normalise to sum=1

print('NNLS weights:')
for name, wi in sorted(zip(names, w), key=lambda x: -x[1]):
    print(f'  {wi*100:5.1f}%  {name}')

# Ensemble predictions
ensemble_preds = (M_te @ w).astype(np.float32)  # (513,)

rae_ens = float(np.mean(np.abs(ensemble_preds[rev_idx] - y_rev)) / baseline)
rho_ens = float(spearmanr(ensemble_preds[rev_idx], y_rev).statistic)
print(f'\nEnsemble (no gate): revealed-RAE={rae_ens:.4f}  ρ={rho_ens:.4f}')

## 6. Activity Gate

For compounds likely to be inactive (classifier score < 0.30), swap the ensemble prediction
with a specialist LGB model trained on RDKit2D descriptors.

> **Optional DrugCLIP enhancement (commented out):**  
> Replace `lgb_rdkit2d_only` gate with `0.7×lgb_rdkit2d_only + 0.3×tabpfn_v3_drugclip_concat`.  
> Requires DrugCLIP model + pocket PDB file; improves gate_RAE by ~0.003.  
> See `optimize_gate.py` and `scripts for drugclip embedding extraction.

### 6a. Activity Classifier (LGB on CheMeleon HTS embeddings, CPU ~5 min)

In [ ]:
CLF_PATH = RES / 'clf_active_ge4_hts_test_preds.npy'

if LOAD_CACHED and CLF_PATH.exists():
    print('Loaded cached classifier')
    clf_preds = np.load(CLF_PATH).flatten()
else:
    # Requires CheMeleon HTS embeddings for the training set
    # Load training embeddings (2048-dim, from model/chemeleon_hts_encoder.pt)
    X_clf_tr = np.load(FEA / 'chemeleon_hts_train.npy').astype(np.float32)  # 4083 × 2048
    X_clf_te = np.load(FEA / 'chemeleon_hts_test.npy').astype(np.float32)   # 513  × 2048
    y_clf    = (y_train >= 4.0).astype(int)

    clf_params = dict(n_estimators=500, learning_rate=0.05, num_leaves=63,
                      subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1)

    seed_probs = []
    for seed in SEEDS:
        folds = scaffold_kfold(smiles_tr, k=KFOLD, seed=seed)
        fold_probs = []
        for tri, vai in folds:
            c = lgb.LGBMClassifier(**{**clf_params, 'random_state': seed})
            c.fit(X_clf_tr[tri], y_clf[tri])
            fold_probs.append(c.predict_proba(X_clf_te)[:, 1])
        seed_probs.append(np.mean(fold_probs, axis=0))

    clf_preds = np.mean(seed_probs, axis=0).astype(np.float32)
    np.save(CLF_PATH, clf_preds)

# Classifier quality on revealed set
inact = y_rev < 3.5
THRESH = 0.30
n_flagged = int((clf_preds[rev_idx] < THRESH).sum())
n_caught  = int((clf_preds[rev_idx][inact] < THRESH).sum())
print(f'Classifier threshold={THRESH}: {n_flagged} compounds flagged  '
      f'({n_caught}/{inact.sum()} true inactives caught)')

### 6b. Gate Model — lgb_rdkit2d_only (CPU ~5 min)

In [ ]:
TAG_LGB  = 'lgb_rdkit2d_only'
TST_LGB  = RES / f'{TAG_LGB}_test_preds.npy'

if LOAD_CACHED and TST_LGB.exists():
    print(f'Loaded cached: {TAG_LGB}')
    lgb_gate_preds = np.load(TST_LGB).flatten()
else:
    from sklearn.preprocessing import StandardScaler
    from sklearn.impute import SimpleImputer

    def preprocess_rdkit2d(Xtr, Xte):
        imp = SimpleImputer(strategy='median')
        Xtr = imp.fit_transform(Xtr).astype(np.float32)
        Xte = imp.transform(Xte).astype(np.float32)
        sc  = StandardScaler()
        Xtr = sc.fit_transform(Xtr).astype(np.float32)
        Xte = sc.transform(Xte).astype(np.float32)
        return Xtr, Xte

    Xtr_lgb, Xte_lgb = preprocess_rdkit2d(X_rd2d_tr.copy(), X_rd2d_te.copy())

    lgb_params = dict(n_estimators=600, learning_rate=0.04, num_leaves=63,
                      subsample=0.8, colsample_bytree=0.8, n_jobs=-1, verbose=-1)

    seed_tests = []
    for seed in SEEDS:
        folds = scaffold_kfold(smiles_tr, k=KFOLD, seed=seed)
        fold_tests = []
        for tri, vai in folds:
            m = lgb.LGBMRegressor(**{**lgb_params, 'random_state': seed})
            m.fit(Xtr_lgb[tri], y_train[tri])
            fold_tests.append(m.predict(Xte_lgb))
        seed_tests.append(np.mean(fold_tests, axis=0))

    lgb_gate_preds = np.mean(seed_tests, axis=0).astype(np.float32)
    np.save(TST_LGB, lgb_gate_preds)

print(f'Gate model ({TAG_LGB}) loaded — inactive-MAE on revealed: '
      f'{np.mean(np.abs(lgb_gate_preds[rev_idx][inact] - y_rev[inact])):.4f}')

# ── Optional DrugCLIP blend (requires DrugCLIP embeddings) ──────────────────
# dc_preds = np.load(RES / 'tabpfn_v3_drugclip_concat_test_preds.npy').flatten()
# lgb_gate_preds = 0.7 * lgb_gate_preds + 0.3 * dc_preds   # improves gate ~0.003 RAE
# ────────────────────────────────────────────────────────────────────────────

### 6c. Apply Gate

In [ ]:
# Gate: replace ensemble predictions with LGB for clf < threshold
gated_preds = ensemble_preds.copy()
gate_mask   = clf_preds < THRESH
gated_preds[gate_mask] = lgb_gate_preds[gate_mask]

n_swapped = int(gate_mask.sum())
rae_gated = float(np.mean(np.abs(gated_preds[rev_idx] - y_rev)) / baseline)
rho_gated = float(spearmanr(gated_preds[rev_idx], y_rev).statistic)

print(f'After gate ({n_swapped} compounds swapped):')
print(f'  revealed-RAE = {rae_gated:.4f}  (was {rae_ens:.4f} before gate)')
print(f'  Spearman ρ   = {rho_gated:.4f}')

## 7. Final Submission

Plug in true pEC50 for the 253 revealed compounds; keep gated model predictions for the 260 blinded.

In [ ]:
pec50_out = gated_preds.copy()

# Overwrite revealed positions with ground-truth
for idx in rev_idx:
    pec50_out[idx] = float(rev_map[mol_names[idx]])

pec50_out = np.clip(pec50_out, 1.0, 9.0).astype(np.float32)

submission = pd.DataFrame({
    'SMILES':        test_df['SMILES'],
    'Molecule Name': test_df['Molecule Name'],
    'pEC50':         pec50_out,
})

out_path = SUB / 'submission_nnls_5model_gate_t030.csv'
submission.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Blinded predictions: [{pec50_out[blind_idx].min():.2f}, {pec50_out[blind_idx].max():.2f}]  '
      f'mean={pec50_out[blind_idx].mean():.3f}')

## 8. Summary

| Step | Output | Notes |
|------|--------|-------|
| Data | 4083 train / 513 test | `data/train_final.csv`, `data/test_curated.csv` |
| Feature extraction | ECFP4+RDKit2D, CheMeleon embeddings | ~5 min GPU |
| tabicl_chemeleon_hts_v2_ecfp4rd | 60% weight | GPU ~30 min (script 145) |
| chemprop_hpo | 19% weight | GPU ~45 min (script 108) |
| graphgps_v3 | 17% weight | GPU ~90 min (script 121) |
| admet_ai | 3% weight | CPU ~10 min |
| tabicl_vanilla_hts | 1% weight | GPU ~10 min (script 24) |
| NNLS weighting | ensemble | Fit on 253 revealed |
| Activity classifier | clf_active_ge4_hts | CPU ~5 min |
| Gate (lgb_rdkit2d_only) | inactive correction | CPU ~5 min |
| **Final submission** | **RAE = 0.5259** | 260 blinded via model, 253 revealed = true values |